## Building the model to predict yield

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, MinMaxScaler, PCA, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.stat import Correlation
from pyspark.mllib.linalg import Vectors
from pyspark.sql import functions as F

In [0]:
sql_query = """SELECT  cy.Year, cy.Country, cy.Item, cy.Element, cy.Unit, cy.Yield, avg_temp.Temperature, corr_index.CPI as CorruptionIndex, sog.GDP, up.UrbanPopulation, up.RuralPopulation, fu.FertilizerUse
FROM agriculture_db.crop_yield cy
INNER JOIN agriculture_db.country_codes cc ON cc.ISONum = cy.AreaCodeM49
INNER JOIN 
(SELECT AVG(ast.Temperature) as Temperature, ast.Year, ast.ISO3
FROM agriculture_db.average_surface_temperature ast
GROUP BY ast.Country, ast.Year, ast.ISO3) avg_temp ON avg_temp.ISO3 = cc.ISO3 AND avg_temp.Year = cy.Year
INNER JOIN agriculture_db.corruption_index corr_index ON corr_index.ISO = cc.ISO3 AND corr_index.Year = cy.Year
INNER JOIN agriculture_db.share_of_gdp sog ON sog.Code = cc.ISO3 AND sog.Year = cy.Year
INNER JOIN agriculture_db.urban_population up ON up.Code = cc.ISO3 AND up.Year = cy.Year
INNER JOIN agriculture_db.fertilizer_use fu ON fu.ISO3 = cc.ISO3 AND fu.Year = cy.Year"""
features_df = spark.sql(sql_query)

In [0]:
# features_df = features_df.withColumn("Yield", F.when(F.isnull(F.col("Yield")), 0).otherwise(F.col("Yield").cast("int")))
features_df = features_df.dropna(subset=["Yield", "CorruptionIndex", "FertilizerUse"])
features_df = (features_df
    .withColumn("Temperature", F.round(F.col("Temperature"), 2))
)
features_df = features_df.where((features_df["Element"] == "Yield"))

In [0]:
country_indexer = StringIndexer(inputCol="Country", outputCol="CountryIndexed", handleInvalid="keep")
item_indexer = StringIndexer(inputCol="Item", outputCol="ItemIndexed", handleInvalid="keep")
country_encoder = OneHotEncoder(inputCol="CountryIndexed", outputCol="CountryEncoded", dropLast=False)
item_encoder = OneHotEncoder(inputCol="ItemIndexed", outputCol="ItemEncoded", dropLast=False)
assembler = VectorAssembler(inputCols=[ "CountryEncoded", "ItemEncoded", "Temperature", "CorruptionIndex", "GDP", "UrbanPopulation", "RuralPopulation", "FertilizerUse"], outputCol="features")

In [0]:
linear_regression_model = LinearRegression(featuresCol="features", labelCol="Yield")

In [0]:
pipeline = Pipeline(stages=[country_indexer, item_indexer, country_encoder, item_encoder, assembler, linear_regression_model])
model = pipeline.fit(features_df)

In [0]:
train_data, test_data = features_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")